# ניסוי ב': כלי AI מול קו הבסיס של OpenCV

שלבים ב'1–ב'3 מתוך `docs/plan.md`: מריצים על 5 תמונות הניסוי את **Real-ESRGAN**, **CodeFormer** (נאמנות `w=0.7`) ו-**Bringing Old Photos Back to Life** (עם `--with_scratch`), ומורידים zip של התוצאות.

**לפני שמתחילים**
1. `Runtime ← Change runtime type ← T4 GPU`.
2. מריצים את התאים **לפי הסדר**. כל כלי מתקין תלויות משלו, ולכן אל תשנו את הסדר (CodeFormer מתקין גרסה משלו של `basicsr`).
3. התמונות עולות ל-VM של Colab. מעלים רק את 5 תמונות הניסוי (יש עליהן הרשאה), ובסוף הסשן התא האחרון מוחק אותן.
4. אם התקנה של כלי נכשלת ולא מצליחים לתקן תוך שעה, מדלגים עליו ורושמים "לא הורץ" בטבלה (כך כתוב בתוכנית).

הקוד נכתב בלי להריץ אותו ב-Colab, ושורות ההתקנה עלולות להישבר בגלל גרסאות חדשות של numpy ו-PyTorch. ראו את "אם משהו נשבר" בסוף.

## 0. הכנה ובדיקת GPU

In [1]:
import os, re, sys, csv, time, shutil, subprocess
from pathlib import Path
from PIL import Image

ROOT = Path("/content")
INPUTS = ROOT / "inputs"
OUT = ROOT / "output_b"
TIMINGS = OUT / "timings.csv"
VERSIONS = OUT / "versions.txt"
EXPECTED = {"1_cracks", "2_stains", "3_tear", "4_blur", "5_faces"}

INPUTS.mkdir(exist_ok=True)
OUT.mkdir(exist_ok=True)
if not TIMINGS.exists():
    TIMINGS.write_text("tool,image,seconds,ok,note\n", encoding="utf-8")

# Old checkpoints fail to load with torch>=2.6 (weights_only=True is the new default)
os.environ["TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD"] = "1"


def log_version(name, repo_dir):
    sha = subprocess.run(["git", "rev-parse", "HEAD"], cwd=repo_dir, capture_output=True, text=True).stdout.strip()
    with open(VERSIONS, "a", encoding="utf-8") as f:
        f.write(f"{name} {sha}\n")
    print(name, sha)


def run_tool(tool, make_cmd, cwd, collect, note=None):
    """Run one process per image (so each gets its own timing, including model load).

    collect(work_dir, image_path) -> {variant_name: result_path}; results are saved as OUT/<variant>/<stem>.png.
    """
    for img in sorted(INPUTS.iterdir()):
        work = ROOT / "work" / tool / img.stem
        if work.exists():
            shutil.rmtree(work)
        (work / "in").mkdir(parents=True)
        shutil.copy(img, work / "in" / img.name)
        start = time.perf_counter()
        proc = subprocess.run(make_cmd(work), cwd=cwd, capture_output=True, text=True)
        seconds = time.perf_counter() - start
        found = collect(work, img) if proc.returncode == 0 else {}
        for variant, path in found.items():
            dest = OUT / variant
            dest.mkdir(exist_ok=True)
            Image.open(path).save(dest / f"{img.stem}.png")
        ok = proc.returncode == 0 and bool(found)
        extra = note(proc) if note and proc.returncode == 0 else ""
        with open(TIMINGS, "a", encoding="utf-8") as f:
            f.write(f"{tool},{img.stem},{seconds:.1f},{ok},{extra}\n")
        print(f"{tool} {img.name}: {'ok' if ok else 'FAILED'} in {seconds:.1f}s {extra}")
        if not ok:
            print((proc.stdout + proc.stderr)[-1500:])


def first_file(folder):
    files = sorted(p for p in Path(folder).glob("*") if p.is_file())
    return files[0] if files else None

In [2]:
import torch

gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU"
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(), "|", gpu)
with open(VERSIONS, "a", encoding="utf-8") as f:
    f.write(f"torch {torch.__version__}; gpu {gpu}\n")

torch 2.11.0+cu130 | cuda: True | Tesla T4


## 1. העלאת 5 התמונות מהמחשב
בוחרים את הקבצים `1_cracks`, `2_stains`, `3_tear`, `4_blur`, `5_faces` (המקוריים, **לא** העותקים המסומנים באדום).

In [3]:
from google.colab import files

uploaded = files.upload()
for name, data in uploaded.items():
    (INPUTS / name).write_bytes(data)

stems = {p.stem for p in INPUTS.iterdir()}
print("uploaded:", sorted(stems))
if stems != EXPECTED:
    print("WARNING - expected exactly:", sorted(EXPECTED))

Saving 1_cracks.jpg to 1_cracks.jpg
Saving 2_stains.jpg to 2_stains.jpg
Saving 3_tear.jpg to 3_tear.jpg
Saving 4_blur.jpg to 4_blur.jpg
Saving 5_faces.jpg to 5_faces.jpg
uploaded: ['1_cracks', '2_stains', '3_tear', '4_blur', '5_faces']


## 2. Real-ESRGAN (שלב ב'1)
`RealESRGAN_x4plus`, **בלי** `--face_enhance` (זה GFPGAN בתוכו ועוקף את ההגדרה השמרנית). הפלט גדול פי 4 מהמקור, וה-`--tile` מונע חריגת זיכרון ב-T4. ההקטנה חזרה לגודל המקור להשוואה תיעשה בשלב ב'4.

In [6]:
%cd /content
!git clone -q https://github.com/xinntao/Real-ESRGAN.git
%cd /content/Real-ESRGAN
!pip install -q basicsr facexlib gfpgan
!pip install -q -r requirements.txt
!pip install -q -e .

# basicsr still imports a torchvision module that newer versions removed: patch it
import importlib.util
spec = importlib.util.find_spec("basicsr")
deg = Path(list(spec.submodule_search_locations)[0]) / "data" / "degradations.py"
deg.write_text(deg.read_text().replace("torchvision.transforms.functional_tensor", "torchvision.transforms.functional"))

log_version("Real-ESRGAN", "/content/Real-ESRGAN")

/content
fatal: destination path 'Real-ESRGAN' already exists and is not an empty directory.
/content/Real-ESRGAN
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This 

AttributeError: 'NoneType' object has no attribute 'submodule_search_locations'

In [ ]:
run_tool(
    "realesrgan",
    lambda w: ["python", "inference_realesrgan.py", "-n", "RealESRGAN_x4plus",
               "-i", str(w / "in"), "-o", str(w / "out"), "--outscale", "4", "--tile", "512"],
    cwd="/content/Real-ESRGAN",
    collect=lambda w, img: {"realesrgan": first_file(w / "out")},
)

## 3. CodeFormer, נאמנות 0.7 (שלב ב'2)
`-w 0.7` נוטה לנאמנות (ערך גבוה = קרוב יותר לקלט), `-s 1` בלי הגדלה, ובלי `--bg_upsampler`. הכלי מזהה פנים בכל התמונה ומשחזר רק אותן. בעמודת `note` נרשם כמה פנים זוהו. ב-`5_faces` בודקים בעין שהפנים לא "התחלפו".

In [ ]:
%cd /content
!git clone -q https://github.com/sczhou/CodeFormer.git
%cd /content/CodeFormer
!pip install -q -r requirements.txt
!python basicsr/setup.py develop
!python scripts/download_pretrained_models.py facelib
!python scripts/download_pretrained_models.py CodeFormer

log_version("CodeFormer", "/content/CodeFormer")

In [ ]:
def faces_note(proc):
    m = re.findall(r"detect (\d+) faces", proc.stdout)
    return f"faces={m[-1]}" if m else "faces=?"


run_tool(
    "codeformer",
    lambda w: ["python", "inference_codeformer.py", "-w", "0.7", "-s", "1",
               "--input_path", str(w / "in"), "--output_path", str(w / "out")],
    cwd="/content/CodeFormer",
    collect=lambda w, img: {"codeformer_w07": first_file(w / "out" / "final_results")},
    note=faces_note,
)

## 4. Bringing Old Photos Back to Life עם `--with_scratch` (שלב ב'3)
הכלי רץ בארבעה שלבים (שחזור גלובלי, זיהוי פנים, שיפור פנים, מיזוג), ולכן ריצה אחת נותנת **שני וריאנטים**:
- `bopbtl_scratch`: פלט שלב 1 (שחזור עם זיהוי שריטות), לפני שיפור הפנים.
- `bopbtl_scratch_face`: הפלט הסופי, אחרי שיפור הפנים.
- `bopbtl_scratch_mask`: מסכת השריטות שהכלי יצר, אם נשמרה.

זה התא הרגיש ביותר: הכלי ישן, ותלויותיו עלולות להתנגש עם Colab של היום. אם לא מצליח תוך שעה, עוצרים ומדלגים.

In [ ]:
%cd /content
!git clone -q https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life.git
%cd /content/Bringing-Old-Photos-Back-to-Life

# SyncBN code both sub-projects need
!git clone -q https://github.com/vacancy/Synchronized-BatchNorm-PyTorch
!cp -rf Synchronized-BatchNorm-PyTorch/sync_batchnorm Face_Enhancement/models/networks/
!cp -rf Synchronized-BatchNorm-PyTorch/sync_batchnorm Global/detection_models/

# dlib landmarks and pretrained checkpoints
%cd Face_Detection
!wget -q http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bzip2 -df shape_predictor_68_face_landmarks.dat.bz2
%cd ../Face_Enhancement
!wget -q https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life/releases/download/v1.0/face_checkpoints.zip
!unzip -oq face_checkpoints.zip
%cd ../Global
!wget -q https://github.com/microsoft/Bringing-Old-Photos-Back-to-Life/releases/download/v1.0/global_checkpoints.zip
!unzip -oq global_checkpoints.zip
%cd ..

!pip install -q -r requirements.txt
log_version("Bringing-Old-Photos", "/content/Bringing-Old-Photos-Back-to-Life")

In [ ]:
BOPBTL = "/content/Bringing-Old-Photos-Back-to-Life"


def bopbtl_collect(w, img):
    out = w / "out"
    found = {
        "bopbtl_scratch": first_file(out / "stage_1_restore_output" / "restored_image"),
        "bopbtl_scratch_face": first_file(out / "final_output"),
        "bopbtl_scratch_mask": first_file(out / "stage_1_restore_output" / "masks" / "mask"),
    }
    return {k: v for k, v in found.items() if v is not None}


run_tool(
    "bopbtl",
    lambda w: ["python", "run.py", "--input_folder", str(w / "in"), "--output_folder", str(w / "out"),
               "--GPU", "0", "--with_scratch"],
    cwd=BOPBTL,
    collect=bopbtl_collect,
)

## 5. סיכום ובדיקה מהירה
כמה תוצרים נוצרו לכל כלי (צפוי 5 לכל אחד), והזמנים שנרשמו.

In [ ]:
for d in sorted(p for p in OUT.iterdir() if p.is_dir()):
    print(f"{d.name}: {len(list(d.glob('*.png')))} files")
print()
print(TIMINGS.read_text(encoding="utf-8"))

## 6. הורדת zip של התוצאות
ה-zip מכיל את `output_b/` (תיקייה לכל כלי, וגם `timings.csv` ו-`versions.txt`). פותחים אותו ב-`experiments/` של הפרויקט, וממשיכים לשלב ב'4.

In [ ]:
archive = shutil.make_archive("/content/output_b", "zip", root_dir="/content", base_dir="output_b")
files.download(archive)

## 7. ניקוי (פרטיות)
מוחק מהסשן את התמונות שהועלו ואת תיקיות העבודה. מריצים רק אחרי שה-zip ירד.

In [ ]:
shutil.rmtree(INPUTS, ignore_errors=True)
shutil.rmtree(ROOT / "work", ignore_errors=True)
print("inputs and work dirs removed")

## אם משהו נשבר
- **`ImportError` מ-`basicsr` או שגיאת numpy:** `Runtime ← Restart session`, מריצים מחדש את תא ההכנה (0) ואת תא ההעלאה, ואז רק את זוג התאים של הכלי שנכשל.
- **`Weights only load failed`:** המשתנה `TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD` נקבע בתא ההכנה. אחרי restart צריך להריץ אותו שוב.
- **Bringing Old Photos נכשל בהתקנה או בהרצה (`np.float`, dlib וכו'):** מדווחים בטבלה "לא הורץ" ואת הסיבה, ולא מתקנים תלויות בלי סוף.
- **קיבוע גרסאות:** `versions.txt` שבתוך ה-zip מכיל את ה-commit של כל כלי. אחרי ריצה מוצלחת אפשר להוסיף `!git checkout <sha>` אחרי כל `git clone`.